# اليوم 5 — Gold والتعافي ومخرجات AI/BI — Labs 07–08

أمل خوتاني — Amal Khotani.

ضمن **هندسة البيانات الحديثة لأنظمة الذكاء الاصطناعي — Modern Data Engineering for AI Systems (SDA-DSC-214)** لدى [أكاديمية سدايا](https://github.com/SDAIAAcademy). #SDAIAAcademy

مواد الدورة ودوالها: **ميعاد المري — Meaad Al-Marri**. البيانات اصطناعية.

هذه نسخة منظمة من قسم اليوم 5 في الدفتر المرفوع `amal_khotani (3).ipynb`؛ حُفظت شيفرة الخلايا الأصلية ومخرجاتها وأرقام تنفيذها كما وردت. لم يُعد تشغيل اللابات أثناء فصل الدفاتر، ولم يُختبر Run all في جلسة نظيفة. أرقام التنفيذ تعود إلى جلسات مختلفة.

قبل إعادة التشغيل، اتبع `docs/SETUP.md` في المستودع: Python 3.11، Java 17، PySpark 3.5.8، Delta Spark 3.3.3 وPy4J 0.10.9.9. يجب أن يكون مستودع المقرر وبياناته ومساحة العمل المطلوبة متاحين؛ وجود المخرجات المحفوظة وحده لا يجهز بيئة التشغيل.

أول خلية تشخيص تعرض محاولة سابقة انتهت بـ FILES_NOT_FOUND، وتليها الاستعادة الناجحة من day04_handoff.zip؛ لا تعني الرسالة القديمة أن الاستعادة اللاحقة فشلت. خلية الاستعادة مخصصة لـ Colab وتتطلب رفع الأرشيف. تأكد من تثبيت حزم Spark المحددة قبل خلية Java 17. أضيفت خلية إعداد بعد Java 17 لاستيراد start_spark وتعريف متغيرات المشروع، وهي غير منفذة في هذه النسخة.


In [4]:
from pathlib import Path
import sys

ROOT = Path("/content/masar-modern-data-engineering")
pointer = ROOT / "outputs/day01_bronze_success.json"

if (ROOT / "src/masar").is_dir() and pointer.is_file():
    sys.path.insert(0, str(ROOT / "src"))
    from masar.workspace import completed_bronze_workspace
    WORK = completed_bronze_workspace(ROOT)
    print("WORKSPACE_READY")
else:
    print("FILES_NOT_FOUND")

FILES_NOT_FOUND


In [5]:
from pathlib import Path
from google.colab import files
import io, os, sys, json, zipfile, subprocess, tempfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]

if len(zip_names) != 1:
    raise ValueError("Please upload only day04_handoff.zip")

ROOT = Path(tempfile.mkdtemp(prefix="masar_restore_", dir="/content")) / "course"

print("Preparing the course repository...", flush=True)
subprocess.run(
    [
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/amalkhotani/masar-modern-data-engineering.git",
        str(ROOT)
    ],
    env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    check=True,
    timeout=180
)

with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as bundle:
    for name in bundle.namelist():
        if not name.startswith("outputs/") or ".." in Path(name).parts:
            raise ValueError("Unexpected archive path: " + name)

    if bundle.testzip() is not None:
        raise ValueError("ZIP integrity check failed")

    bundle.extractall(ROOT)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from masar.workspace import completed_bronze_workspace, require_fixed_dataset
from masar.native_contracts import validate_stage_result

SOURCE = ROOT / "data/masar-small-v1"
require_fixed_dataset(SOURCE)
WORK = completed_bronze_workspace(ROOT)

for stage, filename in [
    ("lab05_streaming", "day04_stream_latest.json"),
    ("lab06_quality", "day04_quality_latest.json")
]:
    report = json.loads((WORK / "reports" / filename).read_text())
    validate_stage_result(stage, report)
    print("Restored report verified:", stage)

print("Python:", sys.version.split()[0])
print("Workspace:", WORK)
print("RESTORED")

Saving day04_handoff.zip to day04_handoff.zip
Preparing the course repository...
Restored report verified: lab05_streaming
Restored report verified: lab06_quality
Python: 3.11.13
Workspace: /content/masar_restore_hrmg5mm2/course/outputs/day01_bronze_xgnqdyt3
RESTORED


day 5  Continue your project

In [11]:
import os
import subprocess
from pathlib import Path

def find_java17():
    return next(
        (p for p in Path("/usr/lib/jvm").glob("java-17-openjdk-*")
         if (p / "bin/java").is_file()),
        None
    )

java_home = find_java17()

if java_home is None:
    print("Installing Java 17...", flush=True)
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True, timeout=180
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"],
        check=True, timeout=300
    )
    java_home = find_java17()

if java_home is None:
    raise RuntimeError("Java 17 installation was not found")

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ["PATH"]

from masar.runtime import require_environment
environment = require_environment()

print(environment["java"])
print("JAVA17_READY")

Installing Java 17...
openjdk version "17.0.20" 2026-07-21
JAVA17_READY


## إعداد اليوم الخامس المستقل

هذه الخلية مضافة عند التنظيم، ولا توجد لها مخرجات تنفيذ. تعالج اعتماد خلايا Gold على استيرادات من أيام سابقة. شغّلها بعد الاستعادة وتهيئة البيئة عند إعادة التنفيذ.


In [ ]:
# Added during organization; not executed in this saved copy.
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Restore the course repository first; see docs/SETUP.md.")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print("Continue workspace:", WORK.relative_to(ROOT))


In [12]:
from masar.serving import run_recovery_exercise

spark = start_spark(WORK, kafka=False)

try:
    result = run_recovery_exercise(spark, SOURCE, WORK)
    validate_stage_result("lab07_gold_recovery", result)

    print(json.dumps({
        "scope": result["scope"],
        "checks": result["checks"]
    }, indent=2))

    print("LAB07_PASSED")
finally:
    spark.stop()

{
  "scope": "DAY05_NATIVE_RECOVERY",
  "checks": {
    "injected_failure_observed": true,
    "previous_release_preserved": true,
    "rebuild_has_new_identity": true,
    "content_equal": true
  }
}
LAB07_PASSED


In [13]:
from masar.serving import run_serving_lab, read_release

spark = start_spark(WORK, kafka=False)

try:
    result = run_serving_lab(spark, SOURCE, WORK)
    validate_stage_result("lab08_serving", result)

    print(json.dumps({
        "scope": result["scope"],
        "checks": result["checks"]
    }, indent=2))

    print("BI totals:", json.dumps(result["bi_summary"], indent=2))

    _, observed_tables = read_release(spark, WORK)

    print(
        "AI feature example:",
        observed_tables["ai.zone_hourly_features"][0]
    )
    print(
        "Future label example:",
        observed_tables["ai.zone_hourly_labels"][0]
    )

    print("LAB08_PASSED")
finally:
    spark.stop()

{
  "scope": "DAY05_NATIVE_SERVING",
  "checks": {
    "gold.zone_hourly_demand_schema_and_keys": true,
    "gold.driver_daily_schema_and_keys": true,
    "bi.dim_zone_schema_and_keys": true,
    "bi.dim_driver_schema_and_keys": true,
    "bi.dim_date_schema_and_keys": true,
    "bi.fact_trips_schema_and_keys": true,
    "ai.zone_hourly_features_schema_and_keys": true,
    "ai.zone_hourly_labels_schema_and_keys": true,
    "fact_grain_75": true,
    "foreign_keys_valid": true,
    "gold_fact_totals_match": true,
    "events_aggregated_before_join": true,
    "group_grains_reconcile": true,
    "labels_not_fabricated": true,
    "feature_availability_checked": true,
    "feature_label_keys_aligned": true
  }
}
BI totals: [
  {
    "zone_key": "Z_DAMMAM",
    "trip_count": 25,
    "total_fare_sar": "670.40"
  },
  {
    "zone_key": "Z_JEDDAH",
    "trip_count": 25,
    "total_fare_sar": "625.20"
  },
  {
    "zone_key": "Z_RIYADH",
    "trip_count": 25,
    "total_fare_sar": "585.00"
  }

In [14]:
import zipfile
from google.colab import files

pointer = ROOT / "outputs/day01_bronze_success.json"
archive = ROOT / "outputs/day05_handoff.zip"

with zipfile.ZipFile(
    archive, "w", compression=zipfile.ZIP_DEFLATED
) as bundle:
    bundle.write(
        pointer,
        arcname=pointer.relative_to(ROOT).as_posix()
    )

    for path in sorted(WORK.rglob("*")):
        if path.is_file():
            bundle.write(
                path,
                arcname=path.relative_to(ROOT).as_posix()
            )

with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None, "ZIP integrity check failed"

print("Saved:", archive.relative_to(ROOT))
files.download(str(archive))

Saved: outputs/day05_handoff.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import json
import zipfile
from masar.native_contracts import validate_stage_result

archive = ROOT / "outputs/day05_handoff.zip"
reports = {}

with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None, "ZIP integrity check failed"

    for stage, filename in [
        ("lab07_gold_recovery", "day05_recovery.json"),
        ("lab08_serving", "day05_serving_latest.json"),
    ]:
        report_path = WORK / "reports" / filename
        report_bytes = report_path.read_bytes()
        report = json.loads(report_bytes)

        validate_stage_result(stage, report)

        archived_path = report_path.relative_to(ROOT).as_posix()
        assert bundle.read(archived_path) == report_bytes, (
            "ZIP needs updating: " + filename
        )

        reports[stage] = report
        print(stage, "VERIFIED — checks:", len(report["checks"]))

gold_pointer = json.loads(
    (WORK / "reports/day05_gold_latest.json").read_text()
)

assert (
    reports["lab08_serving"]["source_release_id"]
    == gold_pointer["run_id"]
), "Lab 08 must use the latest Gold release"

print("DAY05_REPORTS_AND_ZIP_VERIFIED")

lab07_gold_recovery VERIFIED — checks: 4
lab08_serving VERIFIED — checks: 16
DAY05_REPORTS_AND_ZIP_VERIFIED
